# EduVoice — Socratic Tutor Fine-tuning (Gemma 4)
### Cambridge Primary Checkpoint Science (0097) Standard

This notebook implements a high-reasoning **Socratic Tutor** architecture using the **Gemma 4 E4B** model. Unlike standard Q&A bots, this tutor is trained to perform internal analysis before interacting with the student.

### The Architecture: Dual-Block Reasoning
Each model response is structured into two distinct segments:
1. **`<|channel}thought`**: An internal "Examiner Block" where the model analyzes the Cambridge Mark Scheme constraints (e.g., identifying when to reject 'battery' in favor of 'cell').
2. **`<channel|>`**: The "Socratic Nudge" where the model guides the student toward the correct scientific concept without directly revealing the answer.

### Hardware Setup
- **Accelerator:** Tesla T4 (Single)
- **Optimization:** Unsloth (4-bit QLoRA)
- **Memory Footprint:** ~5GB VRAM for Gemma 4 E4B

In [ ]:
%%capture
# 1. Core dependencies for Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

import os
os._exit(0) # Kernel restart to apply paths

In [ ]:
from unsloth import FastLanguageModel
import torch

# Using the Expert (E4B) variant for high-fidelity reasoning
MODEL_NAME = "google/gemma-4-E4b-it"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,       # Auto-detect (Float16 for T4)
    load_in_4bit = True # Fits comfortably in 5GB VRAM
)

# LoRA Config: Targeting all projection layers for maximum reasoning capacity
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", 
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth"
)

print(f"--- {MODEL_NAME} Loaded with LoRA Adapters ---")

In [ ]:
import json
from datasets import Dataset

# Update this path to your exact Kaggle dataset location
DATASET_PATH = "/kaggle/input/datasets/togarayohan/socratic-train/socratic_train.jsonl"

tuning_examples = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        ex = json.loads(line)
        
        # Construct the conversation using the standard Gemma template
        system = ex.get('context', 'You are a Socratic tutor for Cambridge Checkpoint Science.')
        instruction = ex.get('instruction', '')
        response = ex.get('response', '')
        
        formatted_text = (
            f"<start_of_turn>user\n{system}\n\n{instruction}<end_of_turn>\n"
            f"<start_of_turn>model\n{response}<end_of_turn>"
        )
        tuning_examples.append(formatted_text)

dataset = Dataset.from_dict({'text': tuning_examples})
print(f"--- Dataset Ready: {len(dataset)} Socratic Examples ---")

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    tokenizer = tokenizer,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LENGTH,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 1,
        output_dir = './socratic_checkpoints',
        optim = 'adamw_8bit',
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        seed = 3407,
        report_to = 'none',
    )
)

print("--- Starting Socratic Fine-tuning ---")
trainer.train()
print("--- Training Complete ---")

In [ ]:
# Final Verification: Testing the 'Battery vs Cell' trap
FastLanguageModel.for_inference(model)

test_question = "Identify the component in this circuit that provides energy to the bulb. (Student used 'battery')"

prompt = (
    f"<start_of_turn>user\nYou are a Socratic tutor. Adhere to Cambridge 0097 standards. "
    f"Diagram shows one power unit.\n\n{test_question}<end_of_turn>\n"
    f"<start_of_turn>model\n<|channel}}thought\n"
)

inputs = tokenizer(text=[prompt], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer)

print("--- INFERENCE TEST ---")
_ = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 300,
    temperature = 0.3,
    repetition_penalty = 1.2,
    use_cache = True,
    streamer = streamer
)

In [ ]:
# Export to Ollama (GGUF)
model.save_pretrained_gguf("socratic_tutor_gemma4", tokenizer, quantization_method = "q4_k_m")
print("--- Model Exported to GGUF for Offline Use ---")